In [ ]:
%pip install -q semantic-link-labs

In [ ]:
import sempy_labs as labs
from sempy import fabric
import sempy
import pandas
import json
import time

LakehouseName = "SMD_LH"
SemanticModelName = f"{LakehouseName}_model"

In [ ]:
%run 2. Parameters

In [ ]:
lakehouses=labs.list_lakehouses()["Lakehouse Name"]
if LakehouseName in lakehouses.values:
    lakehouseId = notebookutils.lakehouse.getWithProperties(LakehouseName)["id"]
else:
    lakehouseId = fabric.create_lakehouse(LakehouseName)

workspaceId = notebookutils.lakehouse.getWithProperties(LakehouseName)["workspaceId"]
workspaceName = sempy.fabric.resolve_workspace_name(workspaceId)
print(f"WorkspaceId = {workspaceId}, LakehouseID = {lakehouseId}, Workspace Name = {workspaceName}")

In [ ]:
# Build source connection string from parameter file variables
conn_str = f"abfss://{workspaceId}@onelake.dfs.fabric.microsoft.com/{lakehouseId}/Tables"

def loadDataToLakehouse(fromTable: str, toTable: str):
    try:
        # Read from source
        df = spark.read.format("delta").load(f"{conn_str}/{fromTable}")
        
        # Write to target lakehouse (reuse already-resolved workspaceId and lakehouseId)
        (df
         .write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .save(f"abfss://{workspaceId}@onelake.dfs.fabric.microsoft.com/{lakehouseId}/Tables/{toTable}"))
        
        print(f"Loaded {toTable}")
        
    except Exception as e:
        print(f"Error loading {toTable}: {e}")
        raise

tables_to_load = [
    ("agent_dax_documentation", "agent_dax_documentation"),
    ("dim_database", "dim_database"),
    ("table_partitions", "table_partitions"),
    ("diagrams", "diagrams"),
    ("rls", "rls"),
    ("relationships", "relationships"),
    ("column_metadata", "column_metadata"),
    ("data_sources", "data_sources")
]

for from_table, to_table in tables_to_load:
    loadDataToLakehouse(from_table, to_table)

print("Done")

In [ ]:
##https://medium.com/@sqltidy/delays-in-the-automatically-generated-schema-in-the-sql-analytics-endpoint-of-the-lakehouse-b01c7633035d

def triggerMetadataRefresh():
    client = fabric.FabricRestClient()
    response = client.get(f"/v1/workspaces/{workspaceId}/lakehouses/{lakehouseId}")
    sqlendpoint = response.json()['properties']['sqlEndpointProperties']['id']

    # trigger sync
    uri = f"/v1.0/myorg/lhdatamarts/{sqlendpoint}"
    payload = {"commands":[{"$type":"MetadataRefreshExternalCommand"}]}
    response = client.post(uri,json= payload)
    batchId = response.json()['batchId']

    # Monitor Progress
    statusuri = f"/v1.0/myorg/lhdatamarts/{sqlendpoint}/batches/{batchId}"
    statusresponsedata = client.get(statusuri).json()
    progressState = statusresponsedata['progressState']
    print(f"Metadata refresh : {progressState}")
    while progressState != "success":
        statusuri = f"/v1.0/myorg/lhdatamarts/{sqlendpoint}/batches/{batchId}"
        statusresponsedata = client.get(statusuri).json()
        progressState = statusresponsedata['progressState']
        print(f"Metadata refresh : {progressState}")
        time.sleep(1)

    print('Metadata refresh complete')

triggerMetadataRefresh()

In [ ]:
from sempy import fabric

# 1. Only include specific tables in the Semantic Model
lakehouseTables = [
    "agent_dax_documentation",
    "dim_database",
    "column_metadata",
    "diagrams",
    "rls",
    "relationships",
    "data_sources",
    "table_partitions"
]

completedOK: bool = False
while not completedOK:
    try:
        # 2. Create the semantic model
        if sempy.fabric.list_items().query(f"`Display Name`=='{LakehouseName}_model' & Type=='SemanticModel'").shape[0] == 0:
            labs.directlake.generate_direct_lake_semantic_model(
                dataset=f"{LakehouseName}_model",
                lakehouse_tables=lakehouseTables,
                workspace=workspaceName,
                lakehouse=lakehouseId,
                refresh=False,
                overwrite=True
            )
            completedOK = True
    except:
        print('Error creating model... trying again.')
        time.sleep(3)
        triggerMetadataRefresh()

print('Semantic model created OK')

In [ ]:
completedOK: bool = False
while not completedOK:
    try:
        with labs.tom.connect_semantic_model(dataset=SemanticModelName, readonly=False) as tom:

            # 1. Remove any existing relationships
            for r in list(tom.model.Relationships):
                tom.model.Relationships.Remove(r)

            # 2. Create correct relationships (from=Many, to=One)
            tom.add_relationship(from_table="agent_dax_documentation", from_column="DATABASE_VERSION", to_table="dim_database", to_column="DATABASE_VERSION", from_cardinality="Many", to_cardinality="One")
            tom.add_relationship(from_table="diagrams",                from_column="DATABASE_VERSION", to_table="dim_database", to_column="DATABASE_VERSION", from_cardinality="Many", to_cardinality="One")
            tom.add_relationship(from_table="data_sources",            from_column="DATABASE_VERSION", to_table="dim_database", to_column="DATABASE_VERSION", from_cardinality="Many", to_cardinality="One")
            tom.add_relationship(from_table="column_metadata",          from_column="DATABASE_VERSION", to_table="dim_database", to_column="DATABASE_VERSION", from_cardinality="Many", to_cardinality="One")
            tom.add_relationship(from_table="relationships",           from_column="DATABASE_VERSION", to_table="dim_database", to_column="DATABASE_VERSION", from_cardinality="Many", to_cardinality="One")
            tom.add_relationship(from_table="rls",                     from_column="DATABASE_VERSION", to_table="dim_database", to_column="DATABASE_VERSION", from_cardinality="Many", to_cardinality="One")
            tom.add_relationship(from_table="table_partitions",        from_column="DATABASE_VERSION", to_table="dim_database", to_column="DATABASE_VERSION", from_cardinality="Many", to_cardinality="One")
            completedOK = True
    except Exception as e:
        print(f'Error adding relationships: {e} — trying again.')
        time.sleep(3)

print('done')

In [ ]:
with labs.tom.connect_semantic_model(dataset=SemanticModelName, readonly=True) as tom:
    for table in tom.model.Tables:
        print(f"\nTable: {table.Name}")
        for column in table.Columns:
            print(f"  - {column.Name}")

In [ ]:
fabric.refresh_dataset(dataset=SemanticModelName, workspace=workspaceName)